# Bloque 4. XML

**Modulo ETCDFM (5104) - UD2: Formatos de fichero de datos**

---

**XML** (*eXtensible Markup Language*) es el formato de intercambio de datos estructurados mas extendido en sistemas empresariales, servicios web y configuracion de aplicaciones. A diferencia de CSV, XML permite datos **jerarquicos y heterogeneos** con esquema propio. En ML lo encontramos en configuracion de experimentos, feeds de datos y APIs SOAP.


## Contenidos

1. Estructura y sintaxis de XML
2. Parsing basico con `xml.etree.ElementTree`
3. Consultas: `find`, `findall` y XPath
4. Extraer atributos y construir DataFrames
5. Escritura de XML desde Python
6. XML y pandas: `read_xml`, `to_xml`
7. Namespaces en XML
8. Validacion con `lxml` y XSD


## 4.1 Estructura y sintaxis de XML

Un documento XML tiene una estructura de **arbol**:

```xml
<?xml version="1.0" encoding="UTF-8"?>  <!-- declaracion -->
<catalogo version="1.0">               <!-- elemento raiz (root) -->
  <libro id="L001" disponible="true">   <!-- elemento hijo, atributos -->
    <titulo>El quijote</titulo>          <!-- elemento con texto -->
    <autor>Miguel de Cervantes</autor>
    <precio moneda="EUR">12.50</precio>  <!-- atributo + texto -->
  </libro>
</catalogo>
```

**Reglas basicas:**
- Un unico elemento raiz que contiene a todos los demas.
- Etiquetas deben cerrarse: `<tag>...</tag>` o `<tag/>`.
- Los atributos van en la etiqueta de apertura entre comillas.
- El texto de un elemento se llama **text content**.
- Los caracteres especiales se escapan: `&amp;` `&lt;` `&gt;`.


![Anatomía de un documento XML: árbol de elementos](imagenes/b4_anatomia_xml.png)

## 4.2 Parsing basico con ElementTree

`xml.etree.ElementTree` es la libreria de la biblioteca estandar de Python para XML.

```python
import xml.etree.ElementTree as ET

tree = ET.parse('datos/catalogo_libros.xml')  # leer desde fichero
root = tree.getroot()                         # elemento raiz

print(root.tag)            # 'catalogo'
print(root.attrib)         # {'version': '1.0', 'fecha_creacion': '2024-03-01'}
print(len(root))           # numero de hijos directos

for libro in root:         # iterar hijos directos
    print(libro.attrib['id'], libro.find('titulo').text)
```

**API principal:**
- `element.tag` — nombre de la etiqueta
- `element.text` — texto interior
- `element.attrib` — diccionario de atributos
- `element.find(tag)` — primer hijo con ese tag
- `element.findall(tag)` — lista de hijos con ese tag
- `element.iter(tag)` — iterador sobre todos los descendientes


In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse('datos/catalogo_libros.xml')
root = tree.getroot()

print(f'Elemento raiz: {root.tag}')
print(f'Atributos:     {root.attrib}')
print(f'Num. libros:   {len(root)}')
print()

# Iterar los primeros 5 libros
print(f"{'ID':<6} {'Disponible':<12} {'Titulo':<35} {'Autor'}")
print('-' * 80)
for libro in list(root)[:5]:
    libro_id   = libro.attrib['id']
    disp       = libro.attrib['disponible']
    titulo     = libro.find('titulo').text
    autor      = libro.find('autor').text
    print(f'{libro_id:<6} {disp:<12} {titulo:<35} {autor}')

# Acceso a texto y atributos de un elemento especifico
primer_libro = root[0]
precio_elem  = primer_libro.find('precio')
print(f'\nPrecio: {precio_elem.text} {precio_elem.attrib["moneda"]}')


## 4.3 Consultas: `find`, `findall` y XPath

ElementTree soporta un **subconjunto de XPath** para navegar el arbol:

| Expresion XPath | Significado |
|---|---|
| `libro` | Hijos directos llamados `libro` |
| `.//titulo` | Todos los `titulo` en cualquier nivel |
| `libro[@disponible='true']` | Libros donde atributo `disponible='true'` |
| `libro[precio]` | Libros que tienen elemento `precio` |
| `libro[1]` | Primer libro (indice 1-based en XPath) |

```python
# Todos los titulos del catalogo
titulos = root.findall('.//titulo')

# Solo libros disponibles
disponibles = root.findall("libro[@disponible='true']")

# Buscar por condicion de subelemento
tecnicos = root.findall("libro[genero]")
```


In [ ]:
import xml.etree.ElementTree as ET

tree = ET.parse('datos/catalogo_libros.xml')
root = tree.getroot()

# Todos los titulos (cualquier nivel)
titulos = [e.text for e in root.findall('.//titulo')]
print(f'Total titulos: {len(titulos)}')
print(f'Primeros 3: {titulos[:3]}')

# Solo libros disponibles
disponibles = root.findall("libro[@disponible='true']")
print(f'\nLibros disponibles: {len(disponibles)}')

# Libros de genero Tecnico
tecnicos = [l for l in root.findall('libro')
            if l.findtext('genero') == 'Tecnico']
print(f'Libros tecnicos: {len(tecnicos)}')
for l in tecnicos[:3]:
    print(f"  {l.findtext('titulo')[:45]:<45}  {l.findtext('precio')} EUR")

# iter: recorrer todos los precios
precios = [float(e.text) for e in root.iter('precio')]
print(f'\nEstadisticas de precios:')
print(f'  Min:   {min(precios):.2f} EUR')
print(f'  Max:   {max(precios):.2f} EUR')
print(f'  Media: {sum(precios)/len(precios):.2f} EUR')


## 4.4 Extraer datos XML a un DataFrame

El patron habitual es iterar los elementos del nivel relevante y construir un diccionario por cada registro:

```python
import pandas as pd
import xml.etree.ElementTree as ET

root = ET.parse('datos/catalogo_libros.xml').getroot()

registros = []
for libro in root.findall('libro'):
    registros.append({
        'id':         libro.attrib['id'],
        'disponible': libro.attrib['disponible'] == 'true',
        'titulo':     libro.findtext('titulo'),
        'autor':      libro.findtext('autor'),
        'genero':     libro.findtext('genero'),
        'precio':     float(libro.findtext('precio')),
        'stock':      int(libro.findtext('stock')),
    })

df = pd.DataFrame(registros)
```

`findtext(tag)` es equivalente a `find(tag).text` pero devuelve `None` si el elemento no existe.


In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

root = ET.parse('datos/catalogo_libros.xml').getroot()

registros = []
for libro in root.findall('libro'):
    registros.append({
        'id':         libro.attrib['id'],
        'disponible': libro.attrib['disponible'] == 'true',
        'titulo':     libro.findtext('titulo'),
        'autor':      libro.findtext('autor'),
        'genero':     libro.findtext('genero'),
        'precio':     float(libro.findtext('precio')),
        'stock':      int(libro.findtext('stock')),
    })

df = pd.DataFrame(registros)
print(f'Shape: {df.shape}')
print(df.dtypes)
print()
print(df.head(4))
print()
print('Distribucion por genero:')
print(df['genero'].value_counts())
print(f'\nPrecio medio libros disponibles: '
      f'{df[df.disponible]["precio"].mean():.2f} EUR')


## 4.5 Escritura de XML desde Python

ElementTree permite crear documentos XML desde cero con `Element` y `SubElement`:

```python
from xml.etree.ElementTree import Element, SubElement, ElementTree, indent

# Crear la estructura
root = Element('experimento')
root.set('id', 'EXP-001')

modelo = SubElement(root, 'modelo')
modelo.set('tipo', 'random_forest')

SubElement(modelo, 'n_estimators').text = '100'
SubElement(modelo, 'max_depth').text    = '10'

# Indentar (Python >= 3.9)
indent(root)

# Guardar en disco
tree = ElementTree(root)
tree.write('experimento.xml', encoding='utf-8', xml_declaration=True)
```

> `indent(root)` anade saltos de linea y espacios para que el XML sea legible por humanos. Disponible desde Python 3.9.


In [ ]:
from xml.etree.ElementTree import Element, SubElement, ElementTree, indent
import xml.etree.ElementTree as ET

# Leer el XML de experimento existente
tree = ET.parse('datos/experimento_ml.xml')
root = tree.getroot()

print('Experimento:', root.attrib)
print('Nombre:', root.findtext('nombre'))

# Navegar la estructura jerarquica
dataset = root.find('dataset')
print(f"\nDataset: {dataset.attrib['nombre']}")
print(f"  Train: {dataset.findtext('n_muestras/train')}")
print(f"  Test:  {dataset.findtext('n_muestras/test')}")

capas = root.findall('modelo/capas/capa')
print(f"\nCapas del modelo ({len(capas)} total):")
for capa in capas:
    info = dict(capa.attrib)
    print(f"  {info}")

# Crear nuevo XML de experimento desde cero
raiz = Element('experimento')
raiz.set('id', 'EXP-NEW-001')
SubElement(raiz, 'nombre').text = 'Clasificacion con Random Forest'
hp = SubElement(raiz, 'hiperparametros')
SubElement(hp, 'n_estimators').text = '200'
SubElement(hp, 'max_depth').text = '15'
SubElement(hp, 'random_state').text = '42'

indent(raiz)
nuevo_tree = ElementTree(raiz)
nuevo_tree.write('/tmp/nuevo_experimento.xml',
                 encoding='utf-8', xml_declaration=True)

# Verificar
with open('/tmp/nuevo_experimento.xml', 'r', encoding='utf-8') as f:
    print('\nXML generado:')
    print(f.read())


## 4.6 XML y pandas: `read_xml` y `to_xml`

Desde **pandas 1.3** (2021) existe soporte nativo para XML:

```python
# Leer XML directamente a DataFrame
df = pd.read_xml('datos/catalogo_libros.xml',
                  xpath='.//libro')

# Seleccionar campos especificos con XPath
df = pd.read_xml('datos/catalogo_libros.xml',
                  xpath='.//libro',
                  parser='etree')   # o 'lxml' para XPath avanzado

# Exportar DataFrame a XML
df.to_xml('salida.xml',
           root_name='libros',
           row_name='libro',
           index=False)
```

Limitacion: `read_xml` funciona bien con XML **plano** (todos los campos al mismo nivel). Para XML jerarquico complejo (atributos anidados, namespaces) es mejor usar ElementTree directamente.


In [ ]:
import pandas as pd

# Leer el catalogo con pandas
# xpath='.//libro' selecciona todos los elementos <libro>
df = pd.read_xml('datos/catalogo_libros.xml', xpath='.//libro')
print(f'Shape: {df.shape}')
print(df.dtypes)
print()
print(df[['id', 'titulo', 'genero', 'precio']].head(5))

# Analisis rapido
print('\nMedia de precio por genero:')
print(df.groupby('genero')['precio'].agg(['mean','count'])
        .sort_values('mean', ascending=False).round(2))

# Exportar subset a XML
df_tecnicos = df[df['genero'] == 'Tecnico'][['id','titulo','precio']]
df_tecnicos.to_xml('/tmp/libros_tecnicos.xml',
                   root_name='libros_tecnicos',
                   row_name='libro',
                   index=False)

with open('/tmp/libros_tecnicos.xml', 'r', encoding='utf-8') as f:
    lines = f.readlines()
print(f'\nXML exportado ({len(lines)} lineas). Primeras 12:')
print(''.join(lines[:12]))


## 4.7 Namespaces en XML

Los **namespaces** evitan conflictos de nombres cuando se combinan XMLs de distintas fuentes. Se declaran con `xmlns` y se usan con prefijos:

```xml
<rss version="2.0"
     xmlns:dc="http://purl.org/dc/elements/1.1/"
     xmlns:content="http://purl.org/rss/1.0/modules/content/">
  <channel>
    <item>
      <title>Noticia 1</title>
      <dc:creator>Ana Garcia</dc:creator>      <!-- namespace dc -->
      <dc:date>2024-03-10</dc:date>
    </item>
  </channel>
</rss>
```

En ElementTree, los namespaces se representan como `{uri}tag`:

```python
ns = {'dc': 'http://purl.org/dc/elements/1.1/'}

# Con el diccionario de namespaces:
creator = item.find('dc:creator', ns)

# Sin el diccionario (forma larga):
creator = item.find('{http://purl.org/dc/elements/1.1/}creator')
```


In [ ]:
import xml.etree.ElementTree as ET

# Definir los namespaces del RSS
NS = {
    'dc':      'http://purl.org/dc/elements/1.1/',
    'content': 'http://purl.org/rss/1.0/modules/content/',
}

tree = ET.parse('datos/feed_noticias.xml')
root = tree.getroot()

channel = root.find('channel')
print('Feed:', channel.findtext('title'))
print()

# Iterar items del feed
items = channel.findall('item')
print(f'Total noticias: {len(items)}')
print()
print(f"{'Fecha':<12} {'Autor':<22} {'Titular'}")
print('-' * 80)
for item in items:
    titulo  = item.findtext('title', '')
    autor   = item.findtext('dc:creator', '', NS)   # namespace dc
    fecha   = item.findtext('dc:date',    '', NS)
    print(f'{fecha:<12} {autor:<22} {titulo[:44]}')

# Extraer a DataFrame
import pandas as pd
registros = [{
    'fecha':  item.findtext('dc:date', '', NS),
    'autor':  item.findtext('dc:creator', '', NS),
    'titulo': item.findtext('title', ''),
} for item in items]
df_feed = pd.DataFrame(registros)
df_feed['fecha'] = pd.to_datetime(df_feed['fecha'])
print(f'\nDataFrame: {df_feed.shape}')
print(df_feed[['fecha','autor']].head(3))


## 4.8 Validacion con `lxml` y XSD

`lxml` es una libreria externa mas potente que ElementTree: soporta XPath completo, validacion contra DTD/XSD y XSLT.

**XML Schema (XSD)** define la estructura valida de un XML:

```xml
<xs:schema xmlns:xs="http://www.w3.org/2001/XMLSchema">
  <xs:element name="libro">
    <xs:complexType>
      <xs:sequence>
        <xs:element name="titulo" type="xs:string"/>
        <xs:element name="precio" type="xs:decimal"/>
      </xs:sequence>
      <xs:attribute name="id" type="xs:string" use="required"/>
    </xs:complexType>
  </xs:element>
</xs:schema>
```

```python
from lxml import etree

schema  = etree.XMLSchema(file='catalogo_schema.xsd')
doc     = etree.parse('catalogo.xml')
valido  = schema.validate(doc)     # True / False
errores = schema.error_log          # lista de errores si no es valido
```


In [ ]:
try:
    from lxml import etree
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'lxml', '-q',
                    '--break-system-packages'], check=True)
    from lxml import etree

# Validar catalogo contra su XSD
schema_doc = etree.parse('datos/catalogo_schema.xsd')
schema = etree.XMLSchema(schema_doc)

doc = etree.parse('datos/catalogo_libros.xml')
valido = schema.validate(doc)
print(f'catalogo_libros.xml valido: {valido}')
if not valido:
    for error in schema.error_log:
        print(f'  Linea {error.line}: {error.message}')

# Crear un XML invalido y validarlo
xml_invalido = b"""\
<?xml version="1.0" encoding="UTF-8"?>
<catalogo>
  <libro id="X001" disponible="true">
    <titulo>Libro sin precio</titulo>
    <autor>Autor</autor>
    <ano>2024</ano>
    <genero>Tecnico</genero>
    <!-- falta el elemento precio -->
    <stock>5</stock>
  </libro>
</catalogo>"""
doc_inv = etree.fromstring(xml_invalido)
valido_inv = schema.validate(doc_inv)
print(f'\nXML sin <precio> valido: {valido_inv}')
for err in schema.error_log:
    print(f'  Error: {err.message}')

# XPath avanzado con lxml (soporta funciones XPath completas)
root = doc.getroot()
caros = root.xpath("libro[precio > 40]")
print(f'\nLibros con precio > 40 EUR (XPath lxml): {len(caros)}')
for l in caros[:3]:
    titulo = l.find('titulo').text
    precio = l.find('precio').text
    print(f'  {titulo[:40]:<40} {precio} EUR')


## 4.9 Casos de uso de XML en ML y datos

| Ambito | Uso de XML | Ejemplo |
|---|---|---|
| **Configuracion** | Hiperparametros, arquitectura del modelo | `experimento.xml`, Spring, Maven |
| **Datos estructurados** | Catalogos, facturas, historial medico | HL7 FHIR (salud), XBRL (finanzas) |
| **Feeds y APIs** | Sindicacion de contenido, APIs SOAP | RSS/Atom, servicios web empresariales |
| **Anotaciones NLP** | Corpus etiquetados para entrenamiento | TEI, CoNLL-2012, datasets de NER |
| **Graficos vectoriales** | SVG (imagenes como XML) | Datasets de diagramas y esquemas |
| **Intercambio** | Entre sistemas heterogeneos | EDI, OpenDocument, OOXML (.docx) |

**Cuando usar XML vs JSON:**
- XML cuando hay esquema formal (XSD), namespaces o necesidad de validacion rigurosa.
- JSON para APIs REST modernas y configuraciones ligeras.
- CSV/Parquet para datos tabulares.


![Casos de uso de XML en ML y sistemas de datos](imagenes/b4_casos_uso_xml.png)

## Resumen del bloque

**ElementTree** (stdlib) cubre la mayoria de casos:
- `ET.parse()` / `ET.fromstring()` para leer.
- `find()`, `findall()`, `iter()`, XPath basico para consultar.
- `Element()`, `SubElement()`, `indent()`, `write()` para crear y guardar.

**pandas** desde v1.3: `read_xml(xpath=...)` y `to_xml()` para XML plano.

**Namespaces**: mapear prefijos a URIs con un diccionario y pasarlo a `find(tag, ns)`.

**lxml**: mas potente para XPath completo, validacion XSD y XSLT.

---

**Cuando elegir XML:**
datos jerarquicos complejos, esquema formal, integracion con sistemas empresariales o intercambio de datos con validacion estricta.

*Siguiente bloque: JSON y NDJSON - el formato por defecto de las APIs modernas.*
